# Counterfactual Advantage Estimation

Given a branch state $s$ and two sampled continuations $a_A$ (original) and $a_B$ (counterfactual), we:

1. Show the reflector both hindsight trajectories from $s$ and extract forward-looking guidance $g$.
2. Score both actions under two policies:
   - $\pi_\text{base}(a \mid s)$ — the raw actor at the branch state.
   - $\pi_\text{ic}(a \mid s, g)$ — the same actor with $g$ injected into context.
3. Form signed advantage
   $$\Delta = \big[\log \pi_\text{ic}(a_A \mid s, g) - \log \pi_\text{base}(a_A \mid s)\big] - \big[\log \pi_\text{ic}(a_B \mid s, g) - \log \pi_\text{base}(a_B \mid s)\big].$$
4. $\text{sign}(\Delta)$ predicts which action the guided policy prefers. Compare to `preference` (`original` → A, `counterfactual` → B). Report sign-match rate on non-tie pairs.

Scoring uses the **full assistant message** (reasoning + action tag) at `branch_message_index`.

> **vLLM config note.** For the scoring prompts, the base and in-context variants share a long common prefix. If vLLM is launched with `--enable-prefix-caching` (default on in recent versions), the shared prefix is reused across calls and the in-context scoring is essentially free prefill. Confirm with `ps -ef | grep vllm` on the host or by looking for `prefix_cache_hit` in the server logs — this is the single biggest performance lever after prompt length.

In [1]:
import os, json, re, math, random
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import Optional
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from transformers import AutoTokenizer
from tqdm.auto import tqdm

load_dotenv('../configs/vllm.env')

ACTOR_MODEL = os.environ['MODEL_STR']
ACTOR_PORT  = int(os.environ['VLLM_PORT'])
JUDGE_MODEL = os.environ['JUDGE_STR']
JUDGE_PORT  = int(os.environ['JUDGE_PORT'])
VLLM_KEY    = os.environ.get('VLLM_API_KEY', 'local-token')

actor_client = OpenAI(base_url=f'http://localhost:{ACTOR_PORT}/v1', api_key=VLLM_KEY)
judge_client = OpenAI(base_url=f'http://localhost:{JUDGE_PORT}/v1', api_key=VLLM_KEY)

print('actor :', ACTOR_MODEL, 'port', ACTOR_PORT)
print('judge :', JUDGE_MODEL, 'port', JUDGE_PORT)

/home/weiyong/miniconda3/envs/newtonbench/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


actor : Qwen/Qwen3.5-9B port 8002
judge : Qwen/Qwen3.5-35B-A3B-FP8 port 8006


In [2]:
tokenizer = AutoTokenizer.from_pretrained(ACTOR_MODEL)
print('vocab size:', tokenizer.vocab_size)

vocab size: 248044


## Load counterfactual pairs

In [3]:
# Toggle between original judge-based labels and RMSLE-primary labels
USE_RMSLE_LABELS = True  # Set False for original symbolic-equivalence-based labels
USE_AGGREGATED_PAIRS = True  # Set True for 400x3 rollouts (sum RMSLE across same-branch rollouts)

if USE_AGGREGATED_PAIRS:
    label_file = 'counterfactual_pairs_400x3.jsonl'
    df = pd.read_json(label_file, lines=True)

    # Extract per-row RMSLE from nested dicts
    df['orig_rmsle'] = df['original'].apply(lambda x: x['evaluation']['rmsle'])
    df['cf_rmsle']   = df['counterfactual'].apply(lambda x: x['evaluation']['rmsle'])

    # Drop rows with NaN RMSLE (incomplete evaluations)
    df = df.dropna(subset=['orig_rmsle', 'cf_rmsle']).copy()

    # Aggregate RMSLE across rollouts with same pair_index
    agg_orig = df.groupby('pair_index')['orig_rmsle'].first()  # orig is identical per branch
    agg_cf   = df.groupby('pair_index')['cf_rmsle'].sum()       # sum cf RMSLE across rollouts

    # Derive aggregated preference from aggregated RMSLE
    EPSILON = 1e-6
    def _determine_pref(row):
        if abs(row['agg_orig_rmsle'] - row['agg_cf_rmsle']) < EPSILON:
            return 'tie'
        return 'counterfactual' if row['agg_cf_rmsle'] < row['agg_orig_rmsle'] else 'original'

    agg_pref = pd.DataFrame({'agg_orig_rmsle': agg_orig, 'agg_cf_rmsle': agg_cf})
    agg_pref['agg_preference'] = agg_pref.apply(_determine_pref, axis=1)

    # Deduplicate: keep one row per pair_index (rollout with lowest cf_rmsle)
    idx_best = df.groupby('pair_index')['cf_rmsle'].idxmin()
    df = df.loc[idx_best].reset_index(drop=True)

    # Assign aggregated preference, replacing per-row labels
    df = df.merge(agg_pref[['agg_preference']], on='pair_index', how='left')
    df['preference'] = df['agg_preference']
    df = df.drop(columns=['agg_preference', 'orig_rmsle', 'cf_rmsle'])
else:
    label_file = 'counterfactual_pairs_40.jsonl' if USE_RMSLE_LABELS else 'counterfactual_pairs_100.jsonl'
    df = pd.read_json(label_file, lines=True)

df_eval = df[df['preference'] != 'tie'].reset_index(drop=True)
print(f"Label source: {'aggregated-RMSLE' if USE_AGGREGATED_PAIRS else ('RMSLE-primary' if USE_RMSLE_LABELS else 'judge-primary')}")
print(f"Total pairs: {len(df)}, Non-ties: {len(df_eval)}, Ties: {len(df) - len(df_eval)}")
print(f'total={len(df)}  non-tie={len(df_eval)}')
df_eval['preference'].value_counts()



Label source: RMSLE-primary
Total pairs: 114, Non-ties: 70, Ties: 44
total=114  non-tie=70


preference
original          38
counterfactual    32
Name: count, dtype: int64

In [4]:
print(df['equation_difficulty'].value_counts())
print(df['model_system'].value_counts())

print('\nBefore above, after below:\n ')
print(df_eval['equation_difficulty'].value_counts())
df_eval['model_system'].value_counts()

equation_difficulty
easy      47
medium    34
hard      33
Name: count, dtype: int64
model_system
simple_system       47
complex_system      45
vanilla_equation    22
Name: count, dtype: int64

Before above, after below:
 
equation_difficulty
hard      26
easy      25
medium    19
Name: count, dtype: int64


model_system
complex_system      37
simple_system       29
vanilla_equation     4
Name: count, dtype: int64

## Hindsight extraction and reflector

The reflector now sees only the realized non-counterfactual future from the original trajectory and emits forward-looking guidance. Anti-leakage controls remain, but the regeneration filter is intentionally lighter to avoid degenerate rewrite loops. `<think>...</think>` blocks are stripped from assistant messages to cut reflector prompt length.

In [5]:
# Toggle between scoring only the action block vs the full assistant text (without CoT)
SCORE_ACTION_BLOCK_ONLY = True  # Set False to score full assistant text with CoT stripped
USE_JOINT_ACTION_REFLECTOR = True  # Set True to show both actions to reflector and ask for comparison

TASK_HEADER = (
    'TASK CONTEXT:\n'
    'The agent is discovering an unknown physical law in a simulated '
    'universe where physics may differ from ours. Budget: 10 experiment '
    'rounds. Final submission is via <final_law>. Measurements are '
    'noise-free and deterministic. The agent may call <run_experiment>, '
    '<python>, or <final_law> (one action per turn).'
)

_THINK_RE = re.compile(r'<think>.*?</think>\s*', flags=re.DOTALL)
_THINK_TAIL_RE = re.compile(r'^.*?</think>\s*', flags=re.DOTALL)
_FINAL_LAW_RE = re.compile(r'<final_law>.*?</final_law>', flags=re.DOTALL)
MASKED_FINAL_LAW = '<final_law>\n[final law intentionally omitted to avoid answer leakage]\n</final_law>'

GUIDANCE_ISSUE_PATTERNS = {
    'equation_like': re.compile(r'(?:def\s+discovered_law|return\s+|\*\*|\^|[A-Za-z_][A-Za-z0-9_]*\s*[=≈~]\s*[-+]?\d)'),
    'final_law_reference': re.compile(r'(?:<final_law>|discovered_law|functional form)', re.I),
}

def strip_think(content: str) -> str:
    content = _THINK_TAIL_RE.sub('', content)
    content = _THINK_RE.sub('', content)
    if content.lstrip().startswith('Thinking Process:'):
        marker = '\n\n'
        idx = content.find(marker)
        if idx != -1:
            content = content[idx + len(marker):]
    return content.strip()

def mask_final_law_blocks(content: str) -> str:
    return _FINAL_LAW_RE.sub(MASKED_FINAL_LAW, content)

def detect_guidance_issues(content: str) -> list:
    issues = [name for name, pattern in GUIDANCE_ISSUE_PATTERNS.items() if pattern.search(content or '')]
    if len((content or '').split()) > 220:
        issues.append('too_long')
    return issues

def _truncate_by_role(role, content, assistant_len=2000, other_len=600):
    limit = assistant_len if role == 'assistant' else other_len
    if len(content) <= limit:
        return content
    return content[:limit] + f'\n... [truncated, {len(content) - limit} chars omitted]'

def _format_messages(messages, skip_system=True, strip_cot=True, mask_final_law=True):
    lines = []
    for msg in messages:
        role = msg.get('role', 'unknown')
        if skip_system and role == 'system':
            continue
        content = msg.get('content', '')
        if strip_cot and role == 'assistant':
            content = strip_think(content)
        if mask_final_law and role == 'assistant':
            content = mask_final_law_blocks(content)
        content = _truncate_by_role(role, content)
        lines.append(f'[{role.upper()}]:\n{content}\n')
    return '\n'.join(lines)

def extract_action_block(content: str, agent_backend: str = '', mask_final_law: bool = True) -> str:
    for tag in ('final_law', 'run_experiment', 'python'):
        m = re.search(fr'<{tag}>.*?</{tag}>', content, flags=re.DOTALL)
        if m:
            block = m.group(0)
            return mask_final_law_blocks(block) if tag == 'final_law' and mask_final_law else block
    return content.strip()[:300]

def extract_assistant_no_think(content: str, mask_final_law: bool = True) -> str:
    """Return the full assistant text with think blocks removed, optionally masking final law."""
    content = strip_think(content)
    if mask_final_law:
        content = mask_final_law_blocks(content)
    return content.strip()

def extract_hindsight_pointwise(pair: dict) -> dict:
    bi = pair['branch_message_index']
    backend = pair.get('agent_backend', 'vanilla_agent')
    ch_o = pair['original']['chat_history']
    ch_c = pair['counterfactual']['chat_history']
    o_content = ch_o[bi].get('content', '')
    c_content = ch_c[bi].get('content', '')
    return {
        'branch_turn': pair['branch_turn'],
        'branch_message_index': bi,
        'state_prefix': ch_o[:bi],
        'suffix_A': ch_o[bi:],
        'suffix_B': ch_c[bi:],
        'branch_msg_A': ch_o[bi],
        'branch_msg_B': ch_c[bi],
        'action_A': extract_action_block(o_content, backend, mask_final_law=True),
        'action_B': extract_action_block(c_content, backend, mask_final_law=True),
        'score_action_A': (extract_action_block(o_content, backend, mask_final_law=False)
                           if SCORE_ACTION_BLOCK_ONLY else extract_assistant_no_think(o_content, mask_final_law=False)),
        'score_action_B': (extract_action_block(c_content, backend, mask_final_law=False)
                           if SCORE_ACTION_BLOCK_ONLY else extract_assistant_no_think(c_content, mask_final_law=False)),
        'agent_backend': backend,
    }

def format_pointwise_prompt(h: dict, state_window: int = 4, randomize_order: bool = True,
                        seed: Optional[int] = None, strip_cot: bool = True,
                        mask_final_law: bool = True):
    lines = [TASK_HEADER, '', '=' * 60, f"STATE AT DECISION POINT (turn {h['branch_turn']})", '=' * 60]
    lines.append(_format_messages(h['state_prefix'][-state_window:], skip_system=True, strip_cot=strip_cot, mask_final_law=mask_final_law))
    lines.append('=' * 60)
    lines.append('REALIZED FUTURE FROM THIS STATE')
    lines.append('=' * 60)
    lines.append(_format_messages(h['suffix_A'], skip_system=True, strip_cot=strip_cot, mask_final_law=mask_final_law))
    return '\n'.join(lines), {'Realized Future': 'A'}

SYSTEM_MSG_DIRECTIVE = (
    'You are a strategic advisor to a scientific-discovery agent. You observe '
    'the state at a decision point and the realized future that followed from '
    'the action taken there. Your job is to produce guidance that helps the '
    'agent make better decisions at this same state.\n\n'
    'CRITICAL: observing a trajectory does not mean it was a good trajectory. '
    'Judge honestly:\n'
    '- Did the experiments resolve real uncertainty, or did the agent '
    'over-commit on thin evidence?\n'
    '- Were there obvious diagnostics the agent skipped?\n'
    '- Was the final submission premature or well-supported?\n\n'
    'Your guidance will be injected before the agent acts. Be concrete and '
    'directional:\n'
    '- Name specific parameter values, regimes, and experiments from the '
    'trajectory. Quote numbers and ranges when relevant.\n'
    '- Describe observed trends qualitatively: monotonicity, scaling '
    'regimes, where data is thin, where the conclusion is thinly supported.\n'
    '- State whether the evidence supports committing or exploring more.\n'
    '- If exploring, name the single most informative experiment.\n\n'
    'YOU MAY NOT:\n'
    '- Write the functional form of the law as an equation.\n'
    '- State fitted exponents or coefficients.\n'
    '- Produce a <final_law> payload or verbatim experiment JSON.\n\n'
    'Under 200 words. Generic heuristics ("verify carefully", "think step '
    'by step") are actively harmful — omit them.'
)

REFLECTOR_USER_MSG_DIRECTIVE = (
    'Write guidance for the agent about to act at the above state. '
    'Critically assess the realized future: did it build a well-supported '
    'conclusion or over-commit on thin evidence? Recommend a specific next '
    'action and target. Be direct about trajectory quality.'
)

def call_pointwise_reflector(prompt_text: str, temperature: float = 0.4, max_attempts: int = 2) -> dict:
    messages = [
        {'role': 'system', 'content': SYSTEM_MSG_DIRECTIVE},
        {'role': 'user', 'content': f'{prompt_text}\n\n{REFLECTOR_USER_MSG_DIRECTIVE}'},
    ]
    last_raw, last_guidance, last_issues = '', '', []
    for attempt in range(1, max_attempts + 1):
        resp = judge_client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=messages,
            temperature=temperature,
            max_tokens=4096,
            extra_body={'chat_template_kwargs': {'enable_thinking': False}}
        )
        last_raw = resp.choices[0].message.content or ''
        last_guidance = strip_think(last_raw)
        last_issues = detect_guidance_issues(last_guidance)
        if not last_issues:
            return {'guidance': last_guidance, 'raw_guidance': last_raw,
                    'attempts': attempt, 'regenerated': attempt > 1, 'issues': []}
        messages.extend([
            {'role': 'assistant', 'content': last_raw},
            {'role': 'user', 'content': (
                'Rewrite the guidance more simply. Your previous answer violated these constraints: '
                f"{', '.join(last_issues)}. Remove leaked answer content and keep only short strategic guidance."
            )},
        ])
    return {'guidance': last_guidance, 'raw_guidance': last_raw,
            'attempts': max_attempts, 'regenerated': max_attempts > 1, 'issues': last_issues}

SYSTEM_MSG_JOINT_ACTION = (
    'You are a strategic advisor to a scientific-discovery agent at a '
    'decision point. You observe the current state, two candidate actions '
    '(A and B), and the realized future that followed from action A. '
    'Action B was not taken, so you do not observe its future.\n\n'
    'Your job is to compare the two actions and give guidance that helps '
    'the agent choose the better one at this state.\n\n'
    'STRUCTURE YOUR COMPARISON:\n'
    '1. Assess A using the realized future. Did its experiments resolve '
    'real uncertainty, or did A rest on thin or under-probed evidence? '
    'Judge honestly.\n'
    '2. Assess B from its action text alone. What is B probing that the '
    'state has not yet resolved? What assumption does B rest on?\n'
    '3. Take a directional stance. State which action you believe is more '
    'likely to advance correct discovery and why. Do not hedge symmetrically.\n\n'
    'BE CONCRETE AND MATHEMATICALLY EXPLICIT:\n'
    '- Name specific parameter values, regimes, and experiments from the '
    'trajectory. Quote numbers and ranges when relevant.\n'
    '- Explicitly name the mathematical relationships observed (e.g., "the data '
    'suggests an inverse square law" or "y scales linearly with x").\n'
    '- You may use equations to represent structural proportionality (e.g., '
    'y \\propto x^2), but DO NOT provide the exact fitted constants.\n\n'
    'CONSTRAINTS:\n'
    '- DO NOT state the exact final fitted coefficients or the final exact equation.\n'
    '- DO NOT produce a <final_law> payload or verbatim experiment JSON.\n\n'
    'Under 500 words. Generic heuristics ("verify carefully", "think step '
    'by step") are actively harmful — omit them.'
)

REFLECTOR_USER_MSG_JOINT_ACTION = (
    'Compare actions A and B for the agent at the state above. '
    'Assess A from its realized future, assess B from its action text alone, '
    'and state which is more likely to advance correct discovery. '
    'Identify the specific mathematical or logical oversight in the weaker action. '
    'Refer to each action by what it proposes, do not mention actions "A" or "B".'
    'At the end of your answer, output the action A or B in tags <reflector_chosen></reflector_chosen>,'
    'this is the only place where you may mention the action letter.'
)

_REFLECTOR_CHOICE_RE = re.compile(
    r'<reflector_chosen>\s*([AB])\s*</reflector_chosen>',
    flags=re.IGNORECASE,
)

def extract_reflector_choice(guidance: str) -> Optional[str]:
    if not guidance:
        return None
    matches = _REFLECTOR_CHOICE_RE.findall(guidance)
    return matches[-1].upper() if matches else None

def strip_reflector_choice(guidance: str) -> str:
    if not guidance:
        return guidance
    return _REFLECTOR_CHOICE_RE.sub('', guidance).strip()

def format_joint_action_prompt(h: dict, state_window: int = 4,
                               strip_cot: bool = True,
                               mask_final_law: bool = True) -> str:
    lines = [TASK_HEADER, '', '=' * 60,
             f"STATE AT DECISION POINT (turn {h['branch_turn']})", '=' * 60]
    lines.append(_format_messages(
        h['state_prefix'][-state_window:], skip_system=True,
        strip_cot=strip_cot, mask_final_law=mask_final_law))
    lines += ['', '=' * 60,
              'ACTION A (realized future observed below)', '=' * 60,
              h['action_A']]
    lines += ['', '=' * 60,
              'REALIZED FUTURE FROM ACTION A', '=' * 60,
              _format_messages(h['suffix_A'][1:], skip_system=True,
                               strip_cot=strip_cot,
                               mask_final_law=mask_final_law)]
    lines += ['', '=' * 60,
              'ACTION B (future NOT observed; reason from action text only)',
              '=' * 60, h['action_B']]
    return '\n'.join(lines)


def call_joint_action_reflector(prompt_text: str, temperature: float = 0.4, max_attempts: int = 2) -> dict:
    messages = [
        {'role': 'system', 'content': SYSTEM_MSG_JOINT_ACTION},
        {'role': 'user', 'content': f'{prompt_text}\n\n{REFLECTOR_USER_MSG_JOINT_ACTION}'},
    ]
    last_raw, last_guidance, last_issues = '', '', []
    for attempt in range(1, max_attempts + 1):
        resp = judge_client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=messages,
            temperature=temperature,
            max_tokens=4096,  # was 20000 — thinking is off, 4096 is ample
            extra_body={'chat_template_kwargs': {'enable_thinking': False}}
        )
        last_raw = resp.choices[0].message.content or ''
        last_guidance = strip_think(last_raw)
        last_issues = detect_guidance_issues(last_guidance)
        if not last_issues:
            return {'guidance': last_guidance, 'raw_guidance': last_raw,
                    'attempts': attempt, 'regenerated': attempt > 1, 'issues': []}
        messages.extend([
            {'role': 'assistant', 'content': last_raw},
            {'role': 'user', 'content': (
                'Rewrite the guidance more simply. Your previous answer violated these constraints: '
                f"{', '.join(last_issues)}. Remove leaked answer content and keep only short strategic guidance."
            )},
        ])
    return {'guidance': last_guidance, 'raw_guidance': last_raw,
            'attempts': max_attempts, 'regenerated': max_attempts > 1, 'issues': last_issues}


## Scoring: log-probs of a branch assistant message under vLLM

Render the chat with the model's tokenizer, then hit vLLM's `/v1/completions` with `echo=True, logprobs=1` to get per-token logprobs of the prompt. Summing over the assistant-message token span gives $\log \pi(a \mid s)$.

- **Base prompt:** messages up to and including the branch assistant turn.
- **In-context prompt:** same, but with guidance $g$ inserted as a user turn immediately before the branch assistant turn.
- **State windowing:** `SCORING_STATE_WINDOW` optionally trims the pre-branch history to the system message(s) plus the last *N* non-system turns. Base and in-context prompts use the same window so $\Delta$ is still well-defined; this is a speed/fidelity knob. `None` keeps the full prefix.

In [6]:
SCORING_STATE_WINDOW: Optional[int] = None  # None = full history; int = system msgs + last N 
                                                        
GUIDANCE_WRAPPER = (                                                                                
  '[STRATEGIC ADVISOR NOTE — forward-looking guidance for your next action]\n'
  '{g}\n'                                                                                         
  '[END NOTE. Use this context as you decide your next action.]'
)                                                                                                   
                                                        
def _window_state_prefix(prefix, window: Optional[int]):                                            
  if window is None:                                    
      return prefix                                                                               
  system_msgs = [m for m in prefix if m.get('role') == 'system']
  other_msgs  = [m for m in prefix if m.get('role') != 'system']
  if len(other_msgs) <= window:                                                                   
      return prefix
  return system_msgs + other_msgs[-window:]                                                       
                                                        
def build_prompts(state_prefix, branch_msg, guidance: Optional[str],                                
                state_window: Optional[int] = None):    
  state_prefix = _window_state_prefix(state_prefix, state_window)                                 
  msgs = list(state_prefix)
  if guidance is not None:                                                                        
      msgs = msgs + [{'role': 'user', 'content': GUIDANCE_WRAPPER.format(g=guidance)}]
  prefix_text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)   
  full_msgs = msgs + [branch_msg]                                                                 
  full_text = tokenizer.apply_chat_template(full_msgs, tokenize=False,                            
add_generation_prompt=False)                                                                        
  assert full_text.startswith(prefix_text), 'prefix alignment failed — chat template inconsistency'                                                                                      
  return full_text, prefix_text
                                                                                                  
def score_action_logprob(state_prefix, branch_msg, guidance: Optional[str],                         
                       state_window: Optional[int] = None) -> dict:
  full_text, prefix_text = build_prompts(state_prefix, branch_msg, guidance,                      
state_window=state_window)                                                                          
  prefix_ids = tokenizer(prefix_text, add_special_tokens=False)['input_ids']
  full_ids   = tokenizer(full_text,   add_special_tokens=False)['input_ids']                      
  action_len = len(full_ids) - len(prefix_ids)                                                    
  if action_len <= 0:
      return {'logprob': float('nan'), 'n_tokens': 0, 'mean_logprob': float('nan')}               
  resp = actor_client.completions.create(                                                         
      model=ACTOR_MODEL,
      prompt=full_text,                                                                           
      max_tokens=1,                                     
      temperature=0.0,                                                                            
      echo=True,
      logprobs=0,   # was 1 — drops top-k alternatives; token_logprobs still populated            
  )                                                                                               
  tok_logprobs = resp.choices[0].logprobs.token_logprobs
  action_lps = [lp for lp in tok_logprobs[-action_len:] if lp is not None]                        
  if not action_lps:                                    
      return {'logprob': float('nan'), 'n_tokens': 0, 'mean_logprob': float('nan')}               
  return {                                                                                        
      'logprob': float(sum(action_lps)),
      'n_tokens': len(action_lps),                                                                
      'mean_logprob': float(sum(action_lps) / len(action_lps)),                                   
  }
                                                                                                  
def score_action_block_logprob(state_prefix, action_block: str, guidance: Optional[str],            
                            state_window: Optional[int] = None) -> dict:
  action_msg = {'role': 'assistant', 'content': action_block}                                     
  return score_action_logprob(state_prefix, action_msg, guidance, state_window=state_window)      
                                                                                                  

## Token-length diagnostics

Check prompt lengths before running the full sweep. Shows scoring lengths under both full history and `SCORING_STATE_WINDOW` so you can see the speedup the windowing buys.

In [7]:
MAX_CTX = 115_000

def prompt_token_lengths(pair: dict) -> dict:
    h = extract_hindsight_pointwise(pair)
    prompt_text, _ = format_pointwise_prompt(h, randomize_order=False, seed=0)
    reflector_user = f'{prompt_text}\n\n{REFLECTOR_USER_MSG_DIRECTIVE}'
    reflector_total = len(tokenizer(SYSTEM_MSG_DIRECTIVE + reflector_user, add_special_tokens=False)['input_ids'])

    out = {'pair_index': pair['pair_index'], 'reflector_toks': reflector_total}
    for label, branch_msg in [('A', h['branch_msg_A']), ('B', h['branch_msg_B'])]:
        # full (no windowing)
        full_base, _ = build_prompts(h['state_prefix'], branch_msg, guidance=None, state_window=None)
        out[f'{label}_base_full_toks']    = len(tokenizer(full_base, add_special_tokens=False)['input_ids'])
        # windowed (what the sweep will actually send)
        win_base, _ = build_prompts(h['state_prefix'], branch_msg, guidance=None, state_window=SCORING_STATE_WINDOW)
        win_ic,   _ = build_prompts(h['state_prefix'], branch_msg, guidance='[placeholder]', state_window=SCORING_STATE_WINDOW)
        out[f'{label}_base_win_toks']     = len(tokenizer(win_base, add_special_tokens=False)['input_ids'])
        out[f'{label}_ic_win_toks']       = len(tokenizer(win_ic,   add_special_tokens=False)['input_ids'])
    return out

tlen_rows = [prompt_token_lengths(row.to_dict()) for _, row in df_eval.iterrows()]
tlen_df = pd.DataFrame(tlen_rows)

cols = ['reflector_toks',
        'A_base_full_toks', 'A_base_win_toks', 'A_ic_win_toks',
        'B_base_full_toks', 'B_base_win_toks', 'B_ic_win_toks']
print(f'SCORING_STATE_WINDOW = {SCORING_STATE_WINDOW}  (None = full history)')
print(f'{"":28s} {"min":>7} {"p50":>7} {"p90":>7} {"p99":>7} {"max":>7}  over_{MAX_CTX//1000}k')
for col in cols:
    s = tlen_df[col]
    over = (s > MAX_CTX).sum()
    print(f'{col:28s} {s.min():7.0f} {s.quantile(.5):7.0f} {s.quantile(.9):7.0f} {s.quantile(.99):7.0f} {s.max():7.0f}  {over}')

SCORING_STATE_WINDOW = None  (None = full history)
                                 min     p50     p90     p99     max  over_115k
reflector_toks                  3167    8923   14386   18915   18915  0
A_base_full_toks                4425   23308   52231   58728   58728  0
A_base_win_toks                 4425   23308   52231   58728   58728  0
A_ic_win_toks                   4466   23348   52272   58769   58769  0
B_base_full_toks                4657   22446   51536   58574   58574  0
B_base_win_toks                 4657   22446   51536   58574   58574  0
B_ic_win_toks                   4698   22488   51577   58615   58615  0


## Per-pair advantage — flat executor

One shared `EXEC_POOL` runs every vLLM request. Each pair submits the reflector call first, then scores only the extracted action blocks under the guided policy. This matches the writeup's contrastive estimator `log pi(a_A | s, g) - log pi(a_B | s, g)` and avoids scoring the assistant's full reasoning text.

In [8]:
ACTOR_WORKERS = 4    # concurrent actor scoring calls — each holds a 60k-token KV slice
JUDGE_WORKERS = 8    # concurrent reflector calls — judge doesn't OOM on these
COORD_WORKERS = 16   # coordinator threads

ACTOR_POOL = ThreadPoolExecutor(max_workers=ACTOR_WORKERS, thread_name_prefix='actor')
JUDGE_POOL = ThreadPoolExecutor(max_workers=JUDGE_WORKERS, thread_name_prefix='judge')

@dataclass
class PairResult:
    pair_index: int
    preference: str
    gt_sign: int
    guidance: str
    guidance_attempts: int
    guidance_regenerated: bool
    guidance_issues: list
    action_A: str
    action_B: str
    logp_base_A: float
    logp_base_B: float
    logp_ic_A: float
    logp_ic_B: float
    n_tok_A: int
    n_tok_B: int
    base_mean_logp_A: float
    base_mean_logp_B: float
    base_margin: float
    delta: float
    pred_sign: int
    match: bool
    reflector_choice: Optional[str]

def estimate_pair(pair: dict, reflector_seed: Optional[int] = None,
                  state_window: Optional[int] = None) -> PairResult:
    h = extract_hindsight_pointwise(pair)

    if USE_JOINT_ACTION_REFLECTOR:
        prompt_text = format_joint_action_prompt(h, state_window=4)
        fut_ref = JUDGE_POOL.submit(call_joint_action_reflector, prompt_text)
    else:
        prompt_text, _ = format_pointwise_prompt(h, randomize_order=True, seed=reflector_seed)
        fut_ref = JUDGE_POOL.submit(call_pointwise_reflector, prompt_text)

    guidance_info = fut_ref.result()
    guidance = guidance_info['guidance']
    reflector_choice = extract_reflector_choice(guidance)
    guidance = strip_reflector_choice(guidance)

    # Score A: base and ic share state_prefix — prefix cache hit on the long context.
    fut_bA = ACTOR_POOL.submit(score_action_block_logprob, h['state_prefix'], h['score_action_A'], None,     state_window)
    fut_iA = ACTOR_POOL.submit(score_action_block_logprob, h['state_prefix'], h['score_action_A'], guidance, state_window)
    base_A = fut_bA.result()
    ic_A   = fut_iA.result()

    # Score B: sequential after A to halve peak KV usage.
    fut_bB = ACTOR_POOL.submit(score_action_block_logprob, h['state_prefix'], h['score_action_B'], None,     state_window)
    fut_iB = ACTOR_POOL.submit(score_action_block_logprob, h['state_prefix'], h['score_action_B'], guidance, state_window)
    base_B = fut_bB.result()
    ic_B   = fut_iB.result()

    # Hindsight advantage estimator: difference of log-ratios, mean per-token units.
    advantage_A = ic_A['mean_logprob'] - base_A['mean_logprob']
    advantage_B = ic_B['mean_logprob'] - base_B['mean_logprob']
    delta = advantage_A - advantage_B

    gt_sign = +1 if pair['preference'] == 'original' else -1
    pred_sign = 1 if delta > 0 else (-1 if delta < 0 else 0)

    return PairResult(
        pair_index=int(pair['pair_index']),
        preference=pair['preference'],
        gt_sign=gt_sign,
        guidance=guidance,
        guidance_attempts=guidance_info['attempts'],
        guidance_regenerated=guidance_info['regenerated'],
        guidance_issues=guidance_info['issues'],
        action_A=h['score_action_A'],
        action_B=h['score_action_B'],
        logp_base_A=base_A['logprob'],   logp_base_B=base_B['logprob'],
        logp_ic_A=ic_A['logprob'],       logp_ic_B=ic_B['logprob'],
        n_tok_A=ic_A['n_tokens'],         n_tok_B=ic_B['n_tokens'],
        base_mean_logp_A=base_A['mean_logprob'],
        base_mean_logp_B=base_B['mean_logprob'],
        base_margin=base_A['mean_logprob'] - base_B['mean_logprob'],
        delta=delta,
        pred_sign=pred_sign,
        match=(pred_sign == gt_sign),
        reflector_choice=reflector_choice,
    )


## Smoke test on one pair

In [9]:
sample = df_eval.iloc[0].to_dict()                                                                  
h = extract_hindsight_pointwise(sample)                                                             
                                                                                                  
if USE_JOINT_ACTION_REFLECTOR:                                                                      
  prompt_text = format_joint_action_prompt(h, state_window=4)                                     
  guidance_info = call_joint_action_reflector(prompt_text)                                        
else:                                                                                               
  prompt_text, _ = format_pointwise_prompt(h, randomize_order=True, seed=sample['pair_index'])    
  guidance_info = call_pointwise_reflector(prompt_text)                                           

print('pair_index       :', sample['pair_index'])                                                   
print('preference       :', sample['preference'])                                                 
print('reflector mode   :', 'joint_action' if USE_JOINT_ACTION_REFLECTOR else 'pointwise')          
print('attempts         :', guidance_info['attempts'])                                              
print('regenerated      :', guidance_info['regenerated'])
print('issues           :', guidance_info['issues'])                                                
print('\n--- guidance ---\n')                                                                       
print(guidance_info['guidance'])

pair_index       : 0
preference       : counterfactual
reflector mode   : joint_action
attempts         : 2
regenerated      : True
issues           : ['equation_like', 'final_law_reference']

--- guidance ---

Action A succeeded by isolating variables sequentially. It confirmed $F \propto I_1^2$ by doubling $I_1$ and observing a 4x force increase. It confirmed $F \propto 1/I_2$ by doubling $I_2$ and observing a 2x force decrease. It confirmed $F \propto 1/d$ by doubling distance and observing a 2x force decrease. This step-by-step isolation resolved the exponents with high certainty.

Action B proposed running all variations in a single batch. This approach is inefficient because it mixes variables, making it harder to isolate specific exponents without intermediate analysis. It risks wasting the limited budget on redundant data points before the functional form is understood.

Action A is the better choice. Its sequential design allows for immediate hypothesis testing and refinement,

## Run across all non-tie pairs

A coordinator pool dispatches pair-level work; each coordinator is a cheap thread that submits the five vLLM calls into the shared `EXEC_POOL`. With `COORD_WORKERS=32` and `EXEC_WORKERS=16`, up to 16 vLLM requests are in flight at once, globally pipelined across all 48 pairs.

In [10]:
def _run_pair(pair):
    return estimate_pair(pair, reflector_seed=pair['pair_index'], state_window=SCORING_STATE_WINDOW)

print(f"Reflector: {'joint_action' if USE_JOINT_ACTION_REFLECTOR else 'pointwise'}")
print(f"Score target: {'action_block' if SCORE_ACTION_BLOCK_ONLY else 'full_assistant_no_think'}")
print(f"Actor concurrency: {ACTOR_WORKERS}  Judge concurrency: {JUDGE_WORKERS}  Coord workers: {COORD_WORKERS}")

results, errors = [], []
pairs_list = [row.to_dict() for _, row in df_eval.iterrows()]

with ThreadPoolExecutor(max_workers=min(COORD_WORKERS, len(pairs_list)), thread_name_prefix='coord') as coord_pool:
    futures = {coord_pool.submit(_run_pair, pair): pair['pair_index'] for pair in pairs_list}
    for fut in tqdm(as_completed(futures), total=len(futures), desc='pairs'):
        pid = futures[fut]
        try:
            results.append(fut.result())
        except Exception as e:
            errors.append((pid, repr(e)))
            print('error on pair', pid, e)

results.sort(key=lambda r: r.pair_index)
res_df = pd.DataFrame([asdict(r) for r in results])
out_path = Path('adv_est_results.jsonl')
res_df.to_json(out_path, orient='records', lines=True)
print(f'saved {len(res_df)} rows to {out_path}; errors={len(errors)}')

Reflector: joint_action
Score target: action_block
Actor concurrency: 4  Judge concurrency: 8  Coord workers: 16


pairs: 100%|████████████████████████████████████| 70/70 [12:46<00:00, 10.95s/it]

saved 70 rows to adv_est_results.jsonl; errors=0


## Sign-match rate and breakdown

In [11]:
import numpy as np
n = len(res_df)
acc = res_df['match'].mean()
print(f'overall sign-match: {acc:.3f}  (n={n})')
print('\nby ground-truth preference:')
print(res_df.groupby('preference')['match'].agg(['mean', 'count']))
print('\ndelta distribution:')
print(res_df['delta'].describe())
print('\n|delta| on matches vs misses:')
print(res_df.assign(abs_delta=res_df['delta'].abs()).groupby('match')['abs_delta'].describe())

overall sign-match: 0.614  (n=70)

by ground-truth preference:
                    mean  count
preference                     
counterfactual  0.468750     32
original        0.736842     38

delta distribution:
count    70.000000
mean     -0.008897
std       0.084024
min      -0.497770
25%      -0.005824
50%       0.003019
75%       0.018609
max       0.106265
Name: delta, dtype: float64

|delta| on matches vs misses:
       count      mean       std       min       25%       50%       75%  \
match                                                                      
False   27.0  0.023281  0.024938  0.000901  0.003468  0.008426  0.051595   
True    43.0  0.054840  0.088983  0.000227  0.006650  0.018431  0.067264   

            max  
match            
False  0.068762  
True   0.497770  


In [12]:
# quick bootstrap CI on sign-match
rng = np.random.default_rng(0)
B = 2000
boots = [rng.choice(res_df['match'].values, size=len(res_df), replace=True).mean() for _ in range(B)]
lo, hi = np.quantile(boots, [0.025, 0.975])
print(f'sign-match = {acc:.3f}  95% CI [{lo:.3f}, {hi:.3f}]')

sign-match = 0.614  95% CI [0.500, 0.714]


In [13]:
res_df['guidance'].iloc[6]

'Action A failed because it assumed the scaling factor $k$ varies linearly with mass ($k = a + bm$). The realized data showed this produced negative power predictions (e.g., $-134$ vs actual $81$), a physical impossibility. The agent forced a linear fit on a clearly non-linear trend, wasting the turn.\n\nAction B correctly isolates the mass dependence by averaging $k$ for each $m$ and testing a power-law form ($Q \\propto m^{-2.5}$). The data shows $k$ is stable for fixed $m$ but increases with $m$ in a way that fits a power law, not a line. This approach directly probes the correct scaling regime without forcing invalid linear assumptions.\n\nAction B is the correct path. It identifies the specific power-law relationship ($m^{-2.5}$) that Action A missed, avoiding the mathematical error of negative energy.'

In [14]:
# Posthoc filtering by action type and base margin
REQUIRE_SAME_ACTION_TYPE = True
BASE_MARGIN_QUANTILES = [0.5, 0.6, 0.7, 0.8]
SAVE_AUGMENTED_RESULTS = False


def action_type(action_text: str) -> str:
    m = re.search(r'<(run_experiment|python|final_law)>', action_text or '')
    return m.group(1) if m else 'other'


def bootstrap_ci(matches, B: int = 2000, seed: int = 0):
    matches = np.asarray(matches, dtype=float)
    if len(matches) == 0:
        return float('nan'), float('nan')
    rng = np.random.default_rng(seed)
    boots = [rng.choice(matches, size=len(matches), replace=True).mean() for _ in range(B)]
    return np.quantile(boots, [0.025, 0.975])


def summarize_subset(label: str, df_subset: pd.DataFrame, total_n: int):
    n = len(df_subset)
    if n == 0:
        print(f'{label:28s} n=0')
        return
    acc = df_subset['match'].mean()
    lo, hi = bootstrap_ci(df_subset['match'].values)
    print(f'{label:28s} n={n:3d} retained={n / total_n:6.1%} sign-match={acc:.3f} 95% CI [{lo:.3f}, {hi:.3f}]')


def compute_base_scores(pair: dict) -> dict:
    h = extract_hindsight_pointwise(pair)
    fut_A = EXEC_POOL.submit(score_action_block_logprob, h['state_prefix'], h['score_action_A'], None, SCORING_STATE_WINDOW)
    fut_B = EXEC_POOL.submit(score_action_block_logprob, h['state_prefix'], h['score_action_B'], None, SCORING_STATE_WINDOW)
    base_A = fut_A.result()
    base_B = fut_B.result()
    return {
        'pair_index': int(pair['pair_index']),
        'base_mean_logp_A': base_A['mean_logprob'],
        'base_mean_logp_B': base_B['mean_logprob'],
        'base_n_tok_A': base_A['n_tokens'],
        'base_n_tok_B': base_B['n_tokens'],
        'base_margin': base_A['mean_logprob'] - base_B['mean_logprob'],
    }


need_base_recompute = (
    'base_margin' not in res_df.columns
    or 'base_mean_logp_A' not in res_df.columns
    or 'base_mean_logp_B' not in res_df.columns
    or res_df['base_margin'].isna().all()
)

if need_base_recompute:
    print('Recomputing unguided base logprobs for base-margin filtering...')
    pairs_list = [row.to_dict() for _, row in df_eval.iterrows()]
    base_rows = []
    with ThreadPoolExecutor(max_workers=min(COORD_WORKERS, len(pairs_list)), thread_name_prefix='base-coord') as coord_pool:
        futures = {coord_pool.submit(compute_base_scores, pair): pair['pair_index'] for pair in pairs_list}
        for fut in tqdm(as_completed(futures), total=len(futures), desc='base logprobs'):
            base_rows.append(fut.result())
    base_df = pd.DataFrame(base_rows)
    res_df = res_df.drop(columns=['base_mean_logp_A', 'base_mean_logp_B', 'base_n_tok_A', 'base_n_tok_B', 'base_margin'], errors='ignore')
    res_df = res_df.merge(base_df, on='pair_index', how='left')
else:
    print('Using existing base logprobs already present in res_df.')

res_df['action_type_A'] = res_df['action_A'].map(action_type)
res_df['action_type_B'] = res_df['action_B'].map(action_type)
res_df['same_action_type'] = res_df['action_type_A'] == res_df['action_type_B']
res_df['abs_base_margin'] = res_df['base_margin'].abs()

if SAVE_AUGMENTED_RESULTS:
    augmented_out = Path('adv_est_results_with_base.jsonl')
    res_df.to_json(augmented_out, orient='records', lines=True)
    print(f'Saved augmented results to {augmented_out}')

print(f'SCORE_ACTION_BLOCK_ONLY = {SCORE_ACTION_BLOCK_ONLY}')
print(f'REQUIRE_SAME_ACTION_TYPE = {REQUIRE_SAME_ACTION_TYPE}')
print()

all_df = res_df.copy()
summarize_subset('all pairs', all_df, total_n=len(res_df))

same_type_df = all_df[all_df['same_action_type']].copy() if REQUIRE_SAME_ACTION_TYPE else all_df.copy()
label = 'same action type' if REQUIRE_SAME_ACTION_TYPE else 'no type filter'
summarize_subset(label, same_type_df, total_n=len(res_df))

if len(same_type_df) == 0:
    print('No retained pairs after action-type filtering.')
else:
    valid_margin_df = same_type_df.dropna(subset=['abs_base_margin']).copy()
    if len(valid_margin_df) == 0:
        print('No valid base margins available.')
    else:
        print('base-margin sweep within retained pairs:')
        for q in BASE_MARGIN_QUANTILES:
            tau = valid_margin_df['abs_base_margin'].quantile(q)
            filtered_df = valid_margin_df[valid_margin_df['abs_base_margin'] <= tau]
            summarize_subset(f'abs(base_margin) <= q{q:.1f}', filtered_df, total_n=len(res_df))
            print(f'  tau = {tau:.4f}')



Using existing base logprobs already present in res_df.
SCORE_ACTION_BLOCK_ONLY = True
REQUIRE_SAME_ACTION_TYPE = True

all pairs                    n= 70 retained=100.0% sign-match=0.614 95% CI [0.500, 0.714]
same action type             n= 55 retained= 78.6% sign-match=0.564 95% CI [0.436, 0.691]
base-margin sweep within retained pairs:
abs(base_margin) <= q0.5     n= 28 retained= 40.0% sign-match=0.607 95% CI [0.429, 0.786]
  tau = 0.0984
abs(base_margin) <= q0.6     n= 33 retained= 47.1% sign-match=0.576 95% CI [0.394, 0.727]
  tau = 0.1505
abs(base_margin) <= q0.7     n= 38 retained= 54.3% sign-match=0.579 95% CI [0.421, 0.737]
  tau = 0.2043
abs(base_margin) <= q0.8     n= 44 retained= 62.9% sign-match=0.500 95% CI [0.341, 0.636]
  tau = 0.2250


In [15]:
# Compare unguided actor preference vs reflected preference
DEFAULT_BASE_MARGIN_QUANTILES = [0.5, 0.6, 0.7, 0.8]


def _sign(x: float) -> int:
    if pd.isna(x):
        return 0
    return 1 if x > 0 else (-1 if x < 0 else 0)


def _bootstrap_ci(matches, B: int = 2000, seed: int = 0):
    matches = np.asarray(matches, dtype=float)
    if len(matches) == 0:
        return float('nan'), float('nan')
    rng = np.random.default_rng(seed)
    boots = [rng.choice(matches, size=len(matches), replace=True).mean() for _ in range(B)]
    return np.quantile(boots, [0.025, 0.975])


def _ensure_action_type_cols(df: pd.DataFrame) -> pd.DataFrame:
    if 'same_action_type' in df.columns:
        return df

    def action_type(action_text: str) -> str:
        m = re.search(r'<(run_experiment|python|final_law)>', action_text or '')
        return m.group(1) if m else 'other'

    df = df.copy()
    df['action_type_A'] = df['action_A'].map(action_type)
    df['action_type_B'] = df['action_B'].map(action_type)
    df['same_action_type'] = df['action_type_A'] == df['action_type_B']
    return df


def summarize_base_vs_reflected(label: str, df_subset: pd.DataFrame, total_n: int):
    n = len(df_subset)
    if n == 0:
        print(f'{label:28s} n=0')
        return

    base_match = (df_subset['base_pred_sign'] == df_subset['gt_sign'])
    refl_match = (df_subset['pred_sign'] == df_subset['gt_sign'])
    base_acc = base_match.mean()
    refl_acc = refl_match.mean()
    base_lo, base_hi = _bootstrap_ci(base_match.values)
    refl_lo, refl_hi = _bootstrap_ci(refl_match.values)

    helped = ((~base_match) & refl_match).sum()
    hurt = (base_match & (~refl_match)).sum()
    no_flip = (df_subset['base_pred_sign'] == df_subset['pred_sign']).sum()
    flipped_no_gain = ((df_subset['base_pred_sign'] != df_subset['pred_sign']) & (~base_match) & (~refl_match)).sum()

    print(f'{label:28s} n={n:3d} retained={n / total_n:6.1%}')
    print(f'  base sign-match      = {base_acc:.3f} 95% CI [{base_lo:.3f}, {base_hi:.3f}]')
    print(f'  reflected sign-match = {refl_acc:.3f} 95% CI [{refl_lo:.3f}, {refl_hi:.3f}]')
    print(f'  reflection helped    = {helped:3d}')
    print(f'  reflection hurt      = {hurt:3d}')
    print(f'  no sign flip         = {no_flip:3d}')
    print(f'  flip, still wrong    = {flipped_no_gain:3d}')


if 'base_margin' not in res_df.columns:
    raise ValueError('base_margin not found in res_df. Run the previous filtering cell first.')

cmp_df = _ensure_action_type_cols(res_df).copy()
cmp_df['base_pred_sign'] = cmp_df['base_margin'].map(_sign)
cmp_df['abs_base_margin'] = cmp_df['base_margin'].abs()
quantiles = globals().get('BASE_MARGIN_QUANTILES', DEFAULT_BASE_MARGIN_QUANTILES)

print(f'SCORE_ACTION_BLOCK_ONLY = {globals().get("SCORE_ACTION_BLOCK_ONLY", "<unknown>")}')
print(f'REQUIRE_SAME_ACTION_TYPE = {globals().get("REQUIRE_SAME_ACTION_TYPE", "<unknown>")}')
print()

summarize_base_vs_reflected('all pairs', cmp_df, total_n=len(cmp_df))
print()

same_type_df = cmp_df[cmp_df['same_action_type']].copy()
summarize_base_vs_reflected('same action type', same_type_df, total_n=len(cmp_df))

if len(same_type_df) > 0:
    valid_margin_df = same_type_df.dropna(subset=['abs_base_margin']).copy()
    if len(valid_margin_df) > 0:
        print('Base-margin sweep within same-type pairs:')
        for q in quantiles:
            tau = valid_margin_df['abs_base_margin'].quantile(q)
            filtered_df = valid_margin_df[valid_margin_df['abs_base_margin'] <= tau]
            summarize_base_vs_reflected(f'abs(base_margin) <= q{q:.1f}', filtered_df, total_n=len(cmp_df))
            print(f'  tau = {tau:.4f}')
            print()



SCORE_ACTION_BLOCK_ONLY = True
REQUIRE_SAME_ACTION_TYPE = True

all pairs                    n= 70 retained=100.0%
  base sign-match      = 0.486 95% CI [0.371, 0.600]
  reflected sign-match = 0.614 95% CI [0.500, 0.714]
  reflection helped    =  23
  reflection hurt      =  14
  no sign flip         =  33
  flip, still wrong    =   0

same action type             n= 55 retained= 78.6%
  base sign-match      = 0.545 95% CI [0.418, 0.673]
  reflected sign-match = 0.564 95% CI [0.436, 0.691]
  reflection helped    =  13
  reflection hurt      =  12
  no sign flip         =  30
  flip, still wrong    =   0
Base-margin sweep within same-type pairs:
abs(base_margin) <= q0.5     n= 28 retained= 40.0%
  base sign-match      = 0.536 95% CI [0.357, 0.714]
  reflected sign-match = 0.607 95% CI [0.429, 0.786]
  reflection helped    =   9
  reflection hurt      =   7
  no sign flip         =  12
  flip, still wrong    =   0
  tau = 0.0984

abs(base_margin) <= q0.6     n= 33 retained= 47.1%
  base 

In [16]:
cmp_df['gt_prefers_original'] = (cmp_df['preference'] == 'original')
summarize_base_vs_reflected('GT prefers original (= A)',
    cmp_df[cmp_df['gt_prefers_original']], total_n=len(cmp_df))
summarize_base_vs_reflected('GT prefers counterfactual (= B)',
    cmp_df[~cmp_df['gt_prefers_original']], total_n=len(cmp_df))

GT prefers original (= A)    n= 38 retained= 54.3%
  base sign-match      = 0.789 95% CI [0.658, 0.895]
  reflected sign-match = 0.737 95% CI [0.605, 0.868]
  reflection helped    =   8
  reflection hurt      =  10
  no sign flip         =  20
  flip, still wrong    =   0
GT prefers counterfactual (= B) n= 32 retained= 45.7%
  base sign-match      = 0.125 95% CI [0.031, 0.250]
  reflected sign-match = 0.469 95% CI [0.312, 0.625]
  reflection helped    =  15
  reflection hurt      =   4
  no sign flip         =  13
  flip, still wrong    =   0


In [17]:
# Recompute Delta with the correct estimator:
#   Delta = [log pi_ic(a_A | s, g) - log pi_base(a_A | s)]
#         - [log pi_ic(a_B | s, g) - log pi_base(a_B | s)]
# vs. what the code actually computed:
#   Delta_simplified = log pi_ic(a_A | s, g) - log pi_ic(a_B | s, g)

import numpy as np

# Sanity check: required columns present
required = ['logp_ic_A', 'logp_ic_B', 'base_mean_logp_A', 'base_mean_logp_B',
            'n_tok_A', 'n_tok_B', 'gt_sign']
missing = [c for c in required if c not in res_df.columns]
assert not missing, f'Missing columns: {missing}. Run the base-logprob recompute cell first.'

df = res_df.copy()

# logp_ic_* are SUMMED token logprobs; base_mean_logp_* are MEAN per-token.
# Match them by converting ic to per-token means.
df['ic_mean_A'] = df['logp_ic_A'] / df['n_tok_A']
df['ic_mean_B'] = df['logp_ic_B'] / df['n_tok_B']

# Correct estimator (per-token means — scale-consistent across both terms)
df['delta_correct']     = (df['ic_mean_A'] - df['base_mean_logp_A']) \
                        - (df['ic_mean_B'] - df['base_mean_logp_B'])
df['pred_sign_correct'] = np.sign(df['delta_correct']).astype(int)
df['match_correct']     = (df['pred_sign_correct'] == df['gt_sign'])

# Simplified (what the code did) — reconstructed for apples-to-apples comparison
df['delta_simplified']     = df['ic_mean_A'] - df['ic_mean_B']
df['pred_sign_simplified'] = np.sign(df['delta_simplified']).astype(int)
df['match_simplified']     = (df['pred_sign_simplified'] == df['gt_sign'])

# Base-only baseline (no reflection at all)
df['delta_base']     = df['base_mean_logp_A'] - df['base_mean_logp_B']
df['pred_sign_base'] = np.sign(df['delta_base']).astype(int)
df['match_base']     = (df['pred_sign_base'] == df['gt_sign'])

def ci(matches, B=2000, seed=0):
    matches = np.asarray(matches, dtype=float)
    rng = np.random.default_rng(seed)
    boots = [rng.choice(matches, size=len(matches), replace=True).mean() for _ in range(B)]
    return np.quantile(boots, [0.025, 0.975])

def summarize(label, m):
    lo, hi = ci(m.values)
    print(f'  {label:22s} sign-match = {m.mean():.3f}  95% CI [{lo:.3f}, {hi:.3f}]')

print(f'n = {len(df)}')
print('\nOverall:')
summarize('base only',   df['match_base'])
summarize('simplified',  df['match_simplified'])
summarize('CORRECT',     df['match_correct'])

# Same-action-type subset (matching the prior filtered analysis)
if 'same_action_type' in df.columns:
    sub = df[df['same_action_type']]
    print(f'\nSame action type (n={len(sub)}):')
    summarize('base only',   sub['match_base'])
    summarize('simplified',  sub['match_simplified'])
    summarize('CORRECT',     sub['match_correct'])

# How often does the correct estimator disagree with the simplified one?
flip_rate = (df['pred_sign_correct'] != df['pred_sign_simplified']).mean()
print(f'\nPred-sign flips (correct vs simplified): {flip_rate:.1%}')

# Key diagnostic: does correcting the estimator flip signs in the right direction?
helped = ((~df['match_simplified']) & df['match_correct']).sum()
hurt   = (df['match_simplified'] & (~df['match_correct'])).sum()
print(f'  correct helped (was wrong, now right): {helped}')
print(f'  correct hurt   (was right, now wrong): {hurt}')

# GT-split breakdown — tests whether the correction fixes the A-bias
for pref in ['original', 'counterfactual']:
    sub = df[df['preference'] == pref]
    if len(sub) == 0:
        continue
    print(f'\nGT prefers {pref} (n={len(sub)}):')
    summarize('base only',   sub['match_base'])
    summarize('simplified',  sub['match_simplified'])
    summarize('CORRECT',     sub['match_correct'])

n = 70

Overall:
  base only              sign-match = 0.486  95% CI [0.371, 0.600]
  simplified             sign-match = 0.557  95% CI [0.443, 0.671]
  CORRECT                sign-match = 0.614  95% CI [0.500, 0.714]

Same action type (n=55):
  base only              sign-match = 0.545  95% CI [0.418, 0.673]
  simplified             sign-match = 0.600  95% CI [0.473, 0.727]
  CORRECT                sign-match = 0.564  95% CI [0.436, 0.691]

Pred-sign flips (correct vs simplified): 31.4%
  correct helped (was wrong, now right): 13
  correct hurt   (was right, now wrong): 9

GT prefers original (n=38):
  base only              sign-match = 0.789  95% CI [0.658, 0.895]
  simplified             sign-match = 0.921  95% CI [0.816, 1.000]
  CORRECT                sign-match = 0.737  95% CI [0.605, 0.868]

GT prefers counterfactual (n=32):
  base only              sign-match = 0.125  95% CI [0.031, 0.250]
  simplified             sign-match = 0.125  95% CI [0.031, 0.250]
  CORRECT            

In [18]:
# Reflector direct-preference analysis
# Does the reflector, on its own, sign-match against GT?
# If yes: method is still plausible even if actor-scoring is noisy.

df['reflector_pred_sign'] = df['reflector_choice'].map(
    {'A': 1, 'B': -1}
).astype('Int64')

parse_rate = df['reflector_choice'].notna().mean()
print(f'Parse rate: {parse_rate:.1%}  ({df["reflector_choice"].notna().sum()}/{len(df)})')
if parse_rate < 1.0:
    missing = df[df['reflector_choice'].isna()]
    print(f'  {len(missing)} pairs failed to parse — dropped from reflector-only analysis')
    print('  sample of unparsed guidance tails:')
    for _, row in missing.head(3).iterrows():
        tail = (row['guidance'] or '')[-200:]
        print(f'    pair {row["pair_index"]}: ...{tail!r}')

# Restrict to pairs with a valid reflector choice
rdf = df.dropna(subset=['reflector_choice']).copy()
rdf['reflector_match'] = (rdf['reflector_pred_sign'] == rdf['gt_sign'])

summarize('reflector direct', rdf['reflector_match'])
summarize('base only',        rdf['match_base'])
#summarize('simplified',       rdf['match_simplified'])
summarize('CORRECT estimator',rdf['match_correct'])

# GT-split: does the reflector have the same A-bias the base policy does,
# or does it genuinely discriminate?
print('\nGT-split (reflector direct):')
for pref in ['original', 'counterfactual']:
    sub = rdf[rdf['preference'] == pref]
    if len(sub) == 0:
        continue
    print(f'  GT prefers {pref} (n={len(sub)}):')
    summarize('    reflector', sub['reflector_match'])
    summarize('    CORRECT',   sub['match_correct'])

# Agreement between reflector direct-preference and the corrected log-ratio estimator.
# If high: reflector content and actor log-ratios are telling the same story.
# If low: they're tapping different signal sources — interesting either way.
agree = (rdf['reflector_pred_sign'] == rdf['pred_sign_correct']).mean()
print(f'\nAgreement(reflector direct, CORRECT estimator): {agree:.3f}')

# Reflector's standalone A-preference rate — diagnoses analog of base-policy A-bias
refl_a_rate = (rdf['reflector_pred_sign'] == 1).mean()
print(f"Reflector picks A: {refl_a_rate:.1%}  (chance: 50%; "
      f"ground-truth A rate: {(rdf['gt_sign'] == 1).mean():.1%})")

Parse rate: 100.0%  (70/70)
  reflector direct       sign-match = 0.500  95% CI [0.386, 0.614]
  base only              sign-match = 0.486  95% CI [0.371, 0.600]
  CORRECT estimator      sign-match = 0.614  95% CI [0.500, 0.714]

GT-split (reflector direct):
  GT prefers original (n=38):
      reflector          sign-match = 0.763  95% CI [0.605, 0.895]
      CORRECT            sign-match = 0.737  95% CI [0.605, 0.868]
  GT prefers counterfactual (n=32):
      reflector          sign-match = 0.188  95% CI [0.062, 0.313]
      CORRECT            sign-match = 0.469  95% CI [0.312, 0.625]

Agreement(reflector direct, CORRECT estimator): 0.629
Reflector picks A: 78.6%  (chance: 50%; ground-truth A rate: 54.3%)


## Clustered Bootstrap

In [19]:
from collections import Counter

# -- cluster structure --------------------------------------------------
cluster_counts = Counter(df['pair_index'])
n_clusters = len(cluster_counts)
size_dist  = Counter(cluster_counts.values())
print(f'n_rows={len(df)}  n_clusters(pair_index)={n_clusters}')
print('rollouts per cluster:', dict(sorted(size_dist.items())))

# pairs where not all rollouts agree on preference
inconsistent = [
    pid for pid, grp in df.groupby('pair_index')
    if grp['preference'].nunique() > 1
]
print(f'clusters with mixed rollout preferences: {len(inconsistent)}  '
      f'(pair_indexes: {sorted(inconsistent)})')
print()

# -- clustered bootstrap -----------------------------------------------
def clustered_bootstrap_ci(df, match_col, cluster_col='pair_index', B=2000, seed=0):
    rng = np.random.default_rng(seed)
    clusters = df[cluster_col].unique()
    cluster_to_idx = {c: df.index[df[cluster_col] == c].tolist() for c in clusters}
    boots = []
    for _ in range(B):
        sampled = rng.choice(clusters, size=len(clusters), replace=True)
        idx = [i for c in sampled for i in cluster_to_idx[c]]
        boots.append(df.loc[idx, match_col].mean())
    lo, hi = np.quantile(boots, [0.025, 0.975])
    return df[match_col].mean(), lo, hi

# df has match_base / match_simplified / match_correct from the previous cell
print(f"{'estimator':22s}  {'naive CI':20s}  clustered CI  (n_rows={len(df)}, n_clusters={n_clusters})")
print('-' * 72)
for label, col in [('base only', 'match_base'),
                    ('simplified', 'match_simplified'),
                    ('CORRECT', 'match_correct')]:
    acc, clo, chi = clustered_bootstrap_ci(df, col)
    nlo, nhi = ci(df[col].values)   # naive (IID) CI from earlier helper
    print(f'{label:22s}  [{nlo:.3f}, {nhi:.3f}]          [{clo:.3f}, {chi:.3f}]')


n_rows=70  n_clusters(pair_index)=30
rollouts per cluster: {1: 8, 2: 4, 3: 18}
clusters with mixed rollout preferences: 7  (pair_indexes: [3, 10, 11, 16, 18, 33, 34])

estimator               naive CI              clustered CI  (n_rows=70, n_clusters=30)
------------------------------------------------------------------------
base only               [0.371, 0.600]          [0.323, 0.653]
simplified              [0.443, 0.671]          [0.386, 0.712]
CORRECT                 [0.500, 0.714]          [0.493, 0.730]


## Action Breakdown Cell 

Stratifies by action type. 

In [20]:
# Action-type breakdown: does CORRECT's signal come from type-differing pairs, 
# type-same pairs, or both?
#
# Expressiveness-bottleneck hypothesis: if the reflector's guidance operates
# primarily at the action-type level (e.g., "commit now" vs "run another experiment"),
# then sign-match should be higher on type-differing pairs than on type-same pairs
# where both actions are (say) run_experiment with different parameters.

import numpy as np
import pandas as pd

def clustered_bootstrap_ci(df, cluster_col, match_col, B=2000, seed=0):
    """Cluster bootstrap: resample clusters, include all pairs in each sampled cluster."""
    rng = np.random.default_rng(seed)
    clusters = df[cluster_col].unique()
    n_clusters = len(clusters)
    if n_clusters == 0:
        return float('nan'), float('nan'), float('nan')
    cluster_to_idx = {c: df.index[df[cluster_col] == c].tolist() for c in clusters}
    boots = []
    for _ in range(B):
        sampled = rng.choice(clusters, size=n_clusters, replace=True)
        idx = [i for c in sampled for i in cluster_to_idx[c]]
        boots.append(df.loc[idx, match_col].mean())
    lo, hi = np.quantile(boots, [0.025, 0.975])
    return df[match_col].mean(), lo, hi

# Ensure df has action-type and cluster columns
df = _ensure_action_type_cols(df).copy() if '_ensure_action_type_cols' in globals() else df.copy()
cluster_col = 'pair_index'  # swap for 'branch_id' if you have a tighter cluster definition
assert cluster_col in df.columns, f'Need cluster column: {cluster_col}'

# Split counts first so we know if any subset is too small to report
print('=== Composition ===')
print(f'Total pairs: {len(df)}  (unique clusters: {df[cluster_col].nunique()})')
print()
print('Action-type pair breakdown:')
print(df.groupby(['action_type_A', 'action_type_B']).size().sort_values(ascending=False))
print()

# Main split: type-same vs type-differing
same_type    = df[df['same_action_type']].copy()
diff_type    = df[~df['same_action_type']].copy()

print(f'Type-same:       n={len(same_type):3d}  clusters={same_type[cluster_col].nunique():3d}')
print(f'Type-differing:  n={len(diff_type):3d}  clusters={diff_type[cluster_col].nunique():3d}')
print()

# Compare all three estimators across the split
print('=== Sign-match by estimator and type-match status ===')
print(f'{"subset":15s} {"estimator":12s} {"n":>4s}  {"sign-match":>11s}  {"95% CI (clustered)":>24s}')
print('-' * 75)

for name_subset, sub in [('all',            df),
                          ('type-differing', diff_type),
                          ('type-same',      same_type)]:
    if len(sub) == 0:
        continue
    for name_est, col in [('base',       'match_base'),
                          ('simplified', 'match_simplified'),
                          ('CORRECT',    'match_correct')]:
        acc, lo, hi = clustered_bootstrap_ci(sub, cluster_col, col)
        print(f'{name_subset:15s} {name_est:12s} {len(sub):4d}  {acc:11.3f}  [{lo:.3f}, {hi:.3f}]')
    print()

# Same breakdown crossed with GT direction — does the type split behave symmetrically?
print('=== CORRECT estimator: type-match × GT direction ===')
print(f'{"subset":25s} {"n":>4s}  {"sign-match":>11s}  {"95% CI":>20s}')
print('-' * 65)
for name_subset, sub in [('type-differing', diff_type),
                          ('type-same',      same_type)]:
    for pref in ['original', 'counterfactual']:
        ssub = sub[sub['preference'] == pref]
        if len(ssub) == 0:
            continue
        acc, lo, hi = clustered_bootstrap_ci(ssub, cluster_col, 'match_correct')
        label = f'{name_subset}, GT={pref}'
        print(f'{label:25s} {len(ssub):4d}  {acc:11.3f}  [{lo:.3f}, {hi:.3f}]')

# One more: where the signal lives. If reflection helps on type-differing pairs,
# we'd expect the gap (CORRECT - base) to be larger there.
print()
print('=== Reflection lift: CORRECT minus base ===')
for name_subset, sub in [('type-differing', diff_type),
                          ('type-same',      same_type)]:
    if len(sub) == 0:
        continue
    lift = sub['match_correct'].astype(float) - sub['match_base'].astype(float)
    # Per-pair lift is in {-1, 0, +1}; clustered-bootstrap the mean
    rng = np.random.default_rng(0)
    clusters = sub[cluster_col].unique()
    cluster_to_idx = {c: sub.index[sub[cluster_col] == c].tolist() for c in clusters}
    boots = []
    for _ in range(2000):
        sampled = rng.choice(clusters, size=len(clusters), replace=True)
        idx = [i for c in sampled for i in cluster_to_idx[c]]
        boots.append(lift.loc[idx].mean())
    lo, hi = np.quantile(boots, [0.025, 0.975])
    print(f'{name_subset:15s} n={len(sub):3d}  lift={lift.mean():+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]')

=== Composition ===
Total pairs: 70  (unique clusters: 30)

Action-type pair breakdown:
action_type_A   action_type_B 
python          python            37
run_experiment  run_experiment    17
python          run_experiment     8
run_experiment  python             6
final_law       python             1
                final_law          1
dtype: int64

Type-same:       n= 55  clusters= 23
Type-differing:  n= 15  clusters=  7

=== Sign-match by estimator and type-match status ===
subset          estimator       n   sign-match        95% CI (clustered)
---------------------------------------------------------------------------
all             base           70        0.486  [0.323, 0.653]
all             simplified     70        0.557  [0.386, 0.712]
all             CORRECT        70        0.614  [0.493, 0.730]

type-differing  base           15        0.267  [0.000, 0.600]
type-differing  simplified     15        0.400  [0.000, 0.769]
type-differing  CORRECT        15        0.800  [0.

In [ ]:
# RMSLE gap stratification
import numpy as np
import pandas as pd

# Ensure rmsle columns are available — merge from pairs file if missing
    _pairs_df = pd.read_json(label_file, lines=True)
    _rmsle_map = _pairs_df[['pair_index']].copy()
    _rmsle_map['rmsle_A'] = _pairs_df['original'].apply(lambda x: x.get('evaluation', {}).get('rmsle', np.nan))
    _rmsle_map['rmsle_B'] = _pairs_df['counterfactual'].apply(lambda x: x.get('evaluation', {}).get('rmsle', np.nan))
    # pairs file may have multiple rollouts per pair_index; drop to one row per pair
    _rmsle_map = _rmsle_map.drop_duplicates(subset='pair_index', keep='first')
    df = df.merge(_rmsle_map, on='pair_index', how='left')

df['log_rmsle_A'] = np.log(df['rmsle_A'].clip(lower=1e-6))
df['log_rmsle_B'] = np.log(df['rmsle_B'].clip(lower=1e-6))
df['abs_log_rmsle_gap'] = (df['log_rmsle_A'] - df['log_rmsle_B']).abs()

df['gap_quintile'] = pd.qcut(df['abs_log_rmsle_gap'], q=5, labels=False, duplicates='drop')

print(f'{"quintile":>8s}  {"gap range":>20s}  {"n":>4s}  {"sign-match":>11s}  {"95% CI (clustered)":>22s}')
print('-' * 75)
for q in range(5):
    sub = df[df['gap_quintile'] == q]
    if len(sub) == 0: continue
    gap_lo, gap_hi = sub['abs_log_rmsle_gap'].min(), sub['abs_log_rmsle_gap'].max()
    acc, lo, hi = clustered_bootstrap_ci(sub, 'pair_index', 'match_correct')
    print(f'{q:>8d}  [{gap_lo:.2f}, {gap_hi:.2f}]  {len(sub):4d}  {acc:11.3f}  [{lo:.3f}, {hi:.3f}]')

In [ ]:
# Confidence stratification on |Delta| (correct estimator)
df['abs_delta'] = df['delta_correct'].abs()
df['delta_quintile'] = pd.qcut(df['abs_delta'], q=5, labels=False, duplicates='drop')

print(f'{"quintile":>8s}  {"|delta| range":>20s}  {"n":>4s}  {"sign-match":>11s}  {"95% CI":>22s}')
print('-' * 75)
for q in range(5):
    sub = df[df['delta_quintile'] == q]
    if len(sub) == 0: continue
    d_lo, d_hi = sub['abs_delta'].min(), sub['abs_delta'].max()
    acc, lo, hi = clustered_bootstrap_ci(sub, 'pair_index', 'match_correct')
    print(f'{q:>8d}  [{d_lo:.4f}, {d_hi:.4f}]  {len(sub):4d}  {acc:11.3f}  [{lo:.3f}, {hi:.3f}]')